# Extractive Question Answering with PEFT (LoRA) on SQuAD
### Complete all code cells marked with `# TODO` and finish training & inference

**Objective:** Fine-tune a pre-trained BERT model for Extractive QA using:
1. Full fine-tuning (baseline)
2. Hard Freezing (freeze BERT backbone)
3. LoRA (Low-Rank Adaptation) via PEFT
4. LoRA Hyperparameter Tuning

**Dataset:** SQuAD v1.1 (Stanford Question Answering Dataset)

**Model:** `bert-base-uncased`

## Task 1: Environment Setup

In [ ]:
# ==== 1-1-pip: View installed packages ====
# TODO: Enter the command to list installed packages


In [ ]:
# ==== 1-2-install: Install required libraries ====
%pip install transformers==4.44.0 datasets==2.20.0 peft==0.12.0 evaluate accelerate

In [ ]:
# ==== 1-3-verify: Verify installations ====
!pip show transformers peft datasets

## Task 2: Imports & Device Configuration

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import get_peft_model, LoraConfig, TaskType
import evaluate

# ==== 2-1-device: Print training device ====
# TODO: Set device and print it
device = 
print(f'Using device: {device}')


## Task 3: Load & Explore the SQuAD Dataset

In [ ]:
# ==== 3-1-load: Load SQuAD v1.1 dataset ====
# TODO: Use datasets library to load 'squad'
raw_datasets = 


In [ ]:
# ==== 3-2-print: Print dataset structure ====
# TODO: Print the dataset to see splits and features
print()


In [ ]:
# ==== 3-3-sample: Display a sample from training set ====
# TODO: Print the first example showing context, question, and answers
sample = raw_datasets['train'][0]
print(f"Context: {sample['context'][:300]}...")
print(f"Question: {sample['question']}")
print(f"Answer: {sample['answers']}")


In [ ]:
# ==== 3-4-stats: Print dataset statistics ====
# TODO: Print number of training and validation examples
print(f'Training examples: {}')
print(f'Validation examples: {}')


## Task 4: Tokenization & Preprocessing

For extractive QA, we need to find the **start** and **end** token positions of the answer within the context.

In [ ]:
# ==== 4-1-tokenizer: Load tokenizer ====
model_checkpoint = 'bert-base-uncased'
# TODO: Load the tokenizer using AutoTokenizer
tokenizer = 
MAX_LENGTH = 384
STRIDE = 128


In [ ]:
# ==== 4-2-preprocess: Implement preprocessing function ====
def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples['question']]
    
    # TODO: Tokenize questions and contexts with truncation and stride
    inputs = tokenizer(
        ,                          # first input
        ,                          # second input  
        max_length=MAX_LENGTH,
        truncation='only_second',
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length',
    )

    offset_mapping = inputs.pop('offset_mapping')
    sample_map = inputs.pop('overflow_to_sample_mapping')
    answers = examples['answers']
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        # TODO: Get the start and end character positions of the answer
        start_char = 
        end_char = 
        sequence_ids = inputs.sequence_ids(i)

        # Find start and end of context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If answer not in this span, set to CLS
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # TODO: Find the token start and end positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs['start_positions'] = start_positions
    inputs['end_positions'] = end_positions
    return inputs


In [ ]:
# ==== 4-3-apply: Apply preprocessing (use small subset for speed) ====
train_dataset = raw_datasets['train'].select(range(5000))
val_dataset = raw_datasets['validation'].select(range(1000))

# TODO: Apply preprocess_training_examples using .map()
tokenized_train = train_dataset.map(
    ,
    batched=True,
    remove_columns=train_dataset.column_names,
)
tokenized_val = val_dataset.map(
    ,
    batched=True,
    remove_columns=val_dataset.column_names,
)
print(f'Tokenized train: {len(tokenized_train)}, val: {len(tokenized_val)}')


## Task 5: Model Parameter Utilities

In [ ]:
# ==== 5-1-params: Implement parameter counting function ====
def print_model_params(model):
    # TODO: Count total and trainable parameters
    total_params = 
    trainable_params = 
    print(f'Total params: {total_params:,}')
    print(f'Trainable params: {trainable_params:,}')
    print(f'Trainable %: {100 * trainable_params / total_params:.2f}%')
    print(f'Model size: {total_params * 4 / 1024 / 1024:.1f} MB (FP32)')
    return total_params, trainable_params


## Task 6: Full Fine-Tuning (Baseline)

In [ ]:
# ==== 6-1-model: Load model for QA ====
# TODO: Load pre-trained model using AutoModelForQuestionAnswering
model_full = 
model_full.to(device)
print_model_params(model_full)


In [ ]:
# ==== 6-2-train_args: Define training arguments ====
training_args_full = TrainingArguments(
    output_dir='./results_full',
    # TODO: Set the following hyperparameters
    num_train_epochs=,          # recommend 2-3
    per_device_train_batch_size=,  # recommend 8-16
    per_device_eval_batch_size=16,
    learning_rate=,             # recommend 2e-5 to 5e-5
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)


In [ ]:
# ==== 6-3-trainer: Create Trainer and train ====
trainer_full = Trainer(
    model=model_full,
    args=training_args_full,
    # TODO: Pass train and eval datasets
    train_dataset=,
    eval_dataset=,
    data_collator=default_data_collator,
    tokenizer=tokenizer,
)

start_time = time.time()
trainer_full.train()
full_train_time = time.time() - start_time
print(f'Full fine-tuning time: {full_train_time:.1f}s')


## Task 7: Hard Freezing — Freeze BERT Backbone

Freeze the entire BERT encoder and only train the QA output head.

In [ ]:
# ==== 7-1-freeze_model: Load model and freeze backbone ====
model_freeze = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
model_freeze.to(device)

# TODO: Freeze all parameters in the BERT backbone
# Hint: model_freeze.bert contains the backbone
for param in :
    param.requires_grad = 

# Verify: only QA output head should be trainable
print('\n=== After Hard Freezing ===')
print_model_params(model_freeze)


In [ ]:
# ==== 7-2-freeze_train: Train frozen model ====
training_args_freeze = TrainingArguments(
    output_dir='./results_freeze',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,  # higher LR since fewer params
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

trainer_freeze = Trainer(
    model=model_freeze,
    args=training_args_freeze,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=default_data_collator,
    tokenizer=tokenizer,
)

start_time = time.time()
trainer_freeze.train()
freeze_train_time = time.time() - start_time
print(f'Frozen training time: {freeze_train_time:.1f}s')


## Task 8: LoRA (Low-Rank Adaptation) via PEFT

Apply LoRA to the BERT model. LoRA adds small trainable rank-decomposition matrices to attention layers.

In [ ]:
# ==== 8-1-lora_config: Define LoRA configuration ====
model_lora = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
model_lora.to(device)

# TODO: Create LoraConfig
lora_config = LoraConfig(
    task_type=TaskType.QUESTION_ANS,
    r=,                    # rank, recommend 8 or 16
    lora_alpha=,           # scaling factor, recommend 16 or 32
    lora_dropout=,         # dropout, recommend 0.1
    target_modules=[],     # TODO: specify which modules to apply LoRA
                           # Hint: 'query' and 'value' attention layers
)

# TODO: Wrap model with PEFT
model_lora = 
print('\n=== LoRA Model ===')
print_model_params(model_lora)
model_lora.print_trainable_parameters()


In [ ]:
# ==== 8-2-lora_train: Train LoRA model ====
training_args_lora = TrainingArguments(
    output_dir='./results_lora',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    # TODO: Set learning rate for LoRA (typically higher than full FT)
    learning_rate=,         # recommend 1e-4 to 3e-4
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
)

trainer_lora = Trainer(
    model=model_lora,
    args=training_args_lora,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=default_data_collator,
    tokenizer=tokenizer,
)

start_time = time.time()
trainer_lora.train()
lora_train_time = time.time() - start_time
print(f'LoRA training time: {lora_train_time:.1f}s')


## Task 9: LoRA Hyperparameter Tuning

Experiment with different LoRA configurations (rank `r`, `lora_alpha`, `target_modules`) and compare results.

In [ ]:
# ==== 9-1-experiment: Define experiment configurations ====
# TODO: Define at least 3 different LoRA configs to compare
experiments = [
    {'name': 'LoRA-r4',   'r': ,  'alpha': ,  'modules': []},
    {'name': 'LoRA-r8',   'r': ,  'alpha': ,  'modules': []},
    {'name': 'LoRA-r16',  'r': ,  'alpha': ,  'modules': []},
]

results = []

for exp in experiments:
    print(f"\n{'='*50}")
    print(f"Running experiment: {exp['name']}")
    print(f"{'='*50}")
    
    # Load fresh model
    model_exp = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
    model_exp.to(device)
    
    # TODO: Create LoraConfig with experiment params
    config = LoraConfig(
        task_type=TaskType.QUESTION_ANS,
        r=,
        lora_alpha=,
        lora_dropout=0.1,
        target_modules=,
    )
    
    # TODO: Apply PEFT
    model_exp = 
    
    total_p, train_p = print_model_params(model_exp)
    
    args_exp = TrainingArguments(
        output_dir=f'./results_{exp["name"]}',
        num_train_epochs=2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-4,
        weight_decay=0.01,
        evaluation_strategy='epoch',
        logging_steps=50,
        fp16=torch.cuda.is_available(),
    )
    
    trainer_exp = Trainer(
        model=model_exp,
        args=args_exp,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        data_collator=default_data_collator,
        tokenizer=tokenizer,
    )
    
    start = time.time()
    train_result = trainer_exp.train()
    elapsed = time.time() - start
    
    results.append({
        'name': exp['name'],
        'r': exp['r'],
        'alpha': exp['alpha'],
        'trainable_params': train_p,
        'train_loss': train_result.training_loss,
        'time': elapsed,
    })


In [ ]:
# ==== 9-2-results_table: Display comparison table ====
import pandas as pd
df_results = pd.DataFrame(results)
print('\n=== LoRA Hyperparameter Comparison ===')
print(df_results.to_string(index=False))


## Task 10: Comparison Visualization

In [ ]:
# ==== 10-1-bar_chart: Plot trainable params comparison ====
methods = ['Full FT', 'Hard Freeze', 'LoRA (r=8)']
# TODO: Fill in trainable parameter counts from previous tasks
trainable_counts = [, , ]
train_times = [full_train_time, freeze_train_time, lora_train_time]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Trainable Parameters
axes[0].bar(methods, trainable_counts, color=['#e74c3c','#3498db','#2ecc71'])
axes[0].set_title('Trainable Parameters by Method')
axes[0].set_ylabel('Number of Parameters')
axes[0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

# Plot 2: Training Time
axes[1].bar(methods, train_times, color=['#e74c3c','#3498db','#2ecc71'])
axes[1].set_title('Training Time by Method')
axes[1].set_ylabel('Time (seconds)')

plt.tight_layout()
plt.show()


In [ ]:
# ==== 10-2-lora_comparison: Plot LoRA rank comparison ====
# TODO: Create a grouped bar chart comparing LoRA experiments
fig, ax = plt.subplots(figsize=(10, 5))
names = [r['name'] for r in results]
losses = [r['train_loss'] for r in results]
params = [r['trainable_params'] for r in results]

x = np.arange(len(names))
width = 0.35

ax2 = ax.twinx()
ax.bar(x - width/2, losses, width, label='Train Loss', color='#9b59b6')
ax2.bar(x + width/2, params, width, label='Trainable Params', color='#e67e22')

ax.set_xlabel('Experiment')
ax.set_ylabel('Training Loss')
ax2.set_ylabel('Trainable Parameters')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('LoRA Hyperparameter Tuning Results')
plt.tight_layout()
plt.show()


## Task 11: Single Example Inference

In [ ]:
# ==== 11-1-inference: Test QA on a single example ====
def predict_answer(model, tokenizer, question, context):
    # TODO: Tokenize the question-context pair
    inputs = tokenizer(
        ,
        ,
        return_tensors='pt',
        max_length=MAX_LENGTH,
        truncation=True,
        padding=True,
    ).to(device)
    
    model.eval()
    with torch.no_grad():
        # TODO: Get model outputs
        outputs = 
    
    # TODO: Extract start and end logits
    start_logits = 
    end_logits = 
    
    # Find the best answer span
    start_idx = torch.argmax(start_logits).item()
    end_idx = torch.argmax(end_logits).item()
    
    # TODO: Decode the answer tokens
    answer_tokens = inputs['input_ids'][0][start_idx:end_idx+1]
    answer = 
    
    return answer

# Test examples
test_context = '''The Apollo program was the third United States human spaceflight program 
carried out by NASA. The program used Apollo spacecraft and Saturn rockets. 
Apollo 11 was the spaceflight that first landed humans on the Moon on July 20, 1969.'''

test_questions = [
    'Which program first landed humans on the Moon?',
    'When did humans first land on the Moon?',
    'What rockets were used in the Apollo program?',
]

print('=== QA Inference Results ===')
for q in test_questions:
    ans = predict_answer(model_lora, tokenizer, q, test_context)
    print(f'Q: {q}')
    print(f'A: {ans}\n')


## Task 12: Summary & Analysis

Write your analysis comparing the three methods:
1. **Full Fine-Tuning**: All parameters trainable
2. **Hard Freezing**: Only QA head trainable
3. **LoRA**: Low-rank adaptation on attention layers

Consider: trainable parameters, training time, and answer quality.

In [ ]:
# ==== 12-1-summary: Print final summary ====
print('='*60)
print('FINAL COMPARISON SUMMARY')
print('='*60)
print(f'{"Method":<20}{"Trainable Params":<20}{"Train Time (s)":<15}')
print('-'*60)
# TODO: Fill in the values from your experiments
print(f'{"Full Fine-Tune":<20}{"":>15}{full_train_time:>10.1f}')
print(f'{"Hard Freeze":<20}{"":>15}{freeze_train_time:>10.1f}')
print(f'{"LoRA (r=8)":<20}{"":>15}{lora_train_time:>10.1f}')
print('='*60)
